In [1]:
from dataclasses import dataclass
import os
from pathlib import Path

## Entity

In [2]:
@dataclass(frozen=True)
class TrainingConfig:
    root_dir: Path
    trained_model_path: Path
    updated_base_model_path: Path
    training_data: Path
    params_epochs: int
    params_batch_size: int
    params_in_augmentation: bool
    params_image_size: list
    params_learning_rate: float
    params_classes: int
    params_weights: str
    params_include_top: bool

In [3]:
os.getcwd()

'd:\\AI-Food-Recognition-Nutrition-Assistant\\research'

In [4]:
os.chdir("..")

In [5]:
%pwd

'd:\\AI-Food-Recognition-Nutrition-Assistant'

## Configuration Manager

In [6]:
from AI_Food_Recognition_Nutrition_Assistant.constants import *
from AI_Food_Recognition_Nutrition_Assistant.utils.common import read_yaml,create_directories
import tensorflow as tf

In [7]:
class ConfigurationManager:
    def __init__(self,
                 config_filepath = CONFIG_FILE_PATH,
                 params_filepath = PARAMS_FILE_PATH
                 ):
        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)

        create_directories([self.config.artifacts_root])

    def get_training_config(self) -> TrainingConfig:
        config = self.config.training
        params = self.params
        prepare_base_model=self.config.prepare_base_model
        training_data = os.path.join(self.config.data_ingestion.unzip_dir,"food-101","food-101","images")

        create_directories([
            Path(config.root_dir)
        ])

        training_config = TrainingConfig(
            root_dir = Path(config.root_dir),
            trained_model_path = Path(config.trained_model_path),
            updated_base_model_path=Path(prepare_base_model.updated_base_model_path),
            training_data=Path(training_data),
            params_in_augmentation=params.AUGMENTATION,
            params_image_size=params.IMAGE_SIZE,
            params_batch_size=params.BATCH_SIZE,
            params_epochs=params.EPOCHS,
            params_classes= params.CLASSES,
            params_include_top=params.INCLUDE_TOP,
            params_learning_rate=params.LEARNING_RATE,
            params_weights=params.WEIGHTS
        )
            
        return training_config    

## Prepare Training Component

In [8]:
import tensorflow as tf
import time

In [9]:
class Training:
    
    def __init__(self, config: TrainingConfig):
        self.config = config
    
    def get_base_model(self):
        self.model=tf.keras.models.load_model(self.config.updated_base_model_path)

    def train_valid_generator(self):

        datagenerator_kwargs= dict (
            rescale=1./255,
            validation_split=0.20
        )

        dataflow_kwargs=dict(
            target_size=self.config.params_image_size[:-1],
            batch_size=self.config.params_batch_size,
            interpolation="bilinear"
        )

        valid_datagenerator=tf.keras.preprocessing.image.ImageDataGenerator(**datagenerator_kwargs)

        self.valid_generator=valid_datagenerator.flow_from_directory(
            directory=self.config.training_data,
            subset="validation",
            shuffle=False,
            **dataflow_kwargs
        )

        if self.config.params_in_augmentation:
            train_datagenerator=tf.keras.preprocessing.image.ImageDataGenerator(
                rotation_range=40,
                horizontal_flip=True,
                width_shift_range=0.2,
                height_shift_range=0.2,
                zoom_range=0.2,
                shear_range=0.20,
                **datagenerator_kwargs
            )
        else:
            train_datagenerator=valid_datagenerator

        self.train_generator=train_datagenerator.flow_from_directory(
            directory=self.config.training_data,
            subset="training",
            shuffle=True,
            **dataflow_kwargs
        )

    @staticmethod
    def save_model(path:Path, model:tf.keras.Model):
        model.save(path)

    def train(self,callback_list:list):
        self.steps_per_epoch = self.train_generator.samples // self.train_generator.batch_size
        self.validation_steps = self.valid_generator.samples // self.valid_generator.batch_size
        self.model.fit(
            self.train_generator,
            epochs=self.config.params_epochs,
            steps_per_epoch=self.steps_per_epoch,
            validation_data=self.valid_generator,
            validation_steps=self.validation_steps,
            callbacks=callback_list
        )

        self.save_model(
            path=self.config.trained_model_path,
            model=self.model
        )



## Pipeline

In [10]:
from AI_Food_Recognition_Nutrition_Assistant.config.configuration import ConfigurationManager
from AI_Food_Recognition_Nutrition_Assistant.components.prepare_callback import  PrepareCallback

In [11]:
try:
    config = ConfigurationManager()
    prepare_callbacks_config = config.get_prepare_callback_config()
    prepare_callbacks = PrepareCallback(config=prepare_callbacks_config)
    callback_list = prepare_callbacks.get_tb_ckpt_callbacks()

    training_config = config.get_training_config()
    training = Training(config=training_config)
    training.get_base_model()
    training.train_valid_generator()
    training.train(
        callback_list=callback_list
    )

except Exception as e:
    raise e

[2026-03-18 16:51:34,242: INFO: common: yaml file: <_io.TextIOWrapper name='config\\config.yaml' mode='r' encoding='cp1252'> loaded successfully]
[2026-03-18 16:51:34,248: INFO: common: yaml file: <_io.TextIOWrapper name='params.yaml' mode='r' encoding='cp1252'> loaded successfully]
[2026-03-18 16:51:34,249: INFO: common: created directory at: artifacts]
[2026-03-18 16:51:34,251: INFO: common: created directory at: artifacts\prepare_callbacks\checkpoint_dir]
[2026-03-18 16:51:34,252: INFO: common: created directory at: artifacts\prepare_callbacks\tensorboard_log_dir]
[2026-03-18 16:51:34,256: INFO: common: created directory at: artifacts\training]
Found 20200 images belonging to 101 classes.
Found 80800 images belonging to 101 classes.


InvalidArgumentError: Graph execution error:

Detected at node 'categorical_crossentropy/softmax_cross_entropy_with_logits' defined at (most recent call last):
    File "<frozen runpy>", line 198, in _run_module_as_main
    File "<frozen runpy>", line 88, in _run_code
    File "d:\AI-Food-Recognition-Nutrition-Assistant\.venv\Lib\site-packages\ipykernel_launcher.py", line 18, in <module>
      app.launch_new_instance()
    File "d:\AI-Food-Recognition-Nutrition-Assistant\.venv\Lib\site-packages\traitlets\config\application.py", line 1075, in launch_instance
      app.start()
    File "d:\AI-Food-Recognition-Nutrition-Assistant\.venv\Lib\site-packages\ipykernel\kernelapp.py", line 758, in start
      self.io_loop.start()
    File "d:\AI-Food-Recognition-Nutrition-Assistant\.venv\Lib\site-packages\tornado\platform\asyncio.py", line 211, in start
      self.asyncio_loop.run_forever()
    File "C:\Users\adity\anaconda3\Lib\asyncio\base_events.py", line 607, in run_forever
      self._run_o